In [2]:
!pip install scikit-learn
!pip install numpy
!pip install pandas

  Using cached numpy-2.4.3-cp312-cp312-win_amd64.whl.metadata (6.6 kB)
     ---------------------------------------- 0.0/61.0 kB ? eta -:--:--
     ---------------------------------------- 0.0/61.0 kB ? eta -:--:--
     ------ --------------------------------- 10.2/61.0 kB ? eta -:--:--
     ------------------- ------------------ 30.7/61.0 kB 262.6 kB/s eta 0:00:01
     ------------------------- ------------ 41.0/61.0 kB 281.8 kB/s eta 0:00:01
     -------------------------------------- 61.0/61.0 kB 325.1 kB/s eta 0:00:00
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
   ---------------------------------------- 0.0/8.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.0 MB ? eta -:-


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached tzdata-2025.3-py2.py3-none-any.whl.metadata (1.4 kB)
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/9.7 MB 435.7 kB/s eta 0:00:23
   ---------------------------------------- 0.0/9.7 MB 435.7 kB/s eta 0:00:23
   ---------------------------------------- 0.0/9.7 MB 435.7 kB/s eta 0:00:23
   ---------------------------------------- 0.0/9.7 MB 151.3 kB/s eta 0:01:05
   ---------------------------------------- 0.1/9.7 MB 218.8 kB/s eta 0:00:45
   ---------------------------------------- 0.1/9.7 MB 275.8 kB/s eta 0:00:35
   ---------------------------------------- 0.1/9.7 MB 275.8 kB/s eta 0:00:35
    -------------------------------------


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import pandas as pd
import numpy as np
from collections import Counter
import math
import warnings
warnings.filterwarnings('ignore')

In [2]:
# ================================================
#  Helper functions (inchangées)
# ================================================

def entropy(y):
    if len(y) == 0:
        return 0
    p = np.bincount(y) / len(y)
    p = p[p > 0]
    return -np.sum(p * np.log2(p))


def information_gain(X_col, y, threshold):
    parent_entropy = entropy(y)
    left_idx = X_col <= threshold
    right_idx = ~left_idx
    
    if left_idx.sum() == 0 or right_idx.sum() == 0:
        return 0
    
    n = len(y)
    n_l, n_r = left_idx.sum(), right_idx.sum()
    
    gain = parent_entropy - (
        (n_l / n) * entropy(y[left_idx]) +
        (n_r / n) * entropy(y[right_idx])
    )
    return gain


def best_split(X, y, features, min_samples_split=5):
    best_gain = -1
    best_feature = None
    best_threshold = None
    
    for feat_idx in features:
        X_col = X[:, feat_idx]
        possible_thresholds = np.unique(X_col)
        
        if len(possible_thresholds) > 100:
            possible_thresholds = np.quantile(possible_thresholds, np.linspace(0.05, 0.95, 40))
        
        for thresh in possible_thresholds:
            gain = information_gain(X_col, y, thresh)
            if gain > best_gain:
                best_gain = gain
                best_feature = feat_idx
                best_threshold = thresh
                
    return best_feature, best_threshold, best_gain


# ================================================
#  Classes DecisionTree & RandomForest
# ================================================

class DecisionTree:
    def __init__(self, max_depth=12, min_samples_split=20,
                 max_features=None, random_state=None):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.max_features = max_features
        self.random_state = random_state
        self.tree = None
        
    def fit(self, X, y):
        np.random.seed(self.random_state)
        self.tree = self._grow_tree(X, y, depth=0)
        
    def _grow_tree(self, X, y, depth):
        n_samples, n_features = X.shape
        n_labels = len(np.unique(y))
        
        if (depth >= self.max_depth or 
            n_samples < self.min_samples_split or 
            n_labels == 1):
            leaf_value = Counter(y).most_common(1)[0][0]
            return {"leaf": True, "value": leaf_value}
        
        if self.max_features is None:
            feat_subset = range(n_features)
        else:
            feat_subset = np.random.choice(range(n_features), 
                                           size=self.max_features, 
                                           replace=False)
        
        best_feat, best_thresh, best_gain = best_split(X, y, feat_subset, self.min_samples_split)
        
        if best_gain <= 0:
            leaf_value = Counter(y).most_common(1)[0][0]
            return {"leaf": True, "value": leaf_value}
        
        left_idx = X[:, best_feat] <= best_thresh
        right_idx = ~left_idx
        
        left_subtree = self._grow_tree(X[left_idx], y[left_idx], depth + 1)
        right_subtree = self._grow_tree(X[right_idx], y[right_idx], depth + 1)
        
        return {
            "leaf": False,
            "feature": best_feat,
            "threshold": best_thresh,
            "left": left_subtree,
            "right": right_subtree
        }
    
    def predict_one(self, x):
        node = self.tree
        while not node.get("leaf", False):
            if x[node["feature"]] <= node["threshold"]:
                node = node["left"]
            else:
                node = node["right"]
        return node["value"]
    
    def predict(self, X):
        return np.array([self.predict_one(x) for x in X])


class RandomForest:
    def __init__(self, n_trees=300, max_depth=12, min_samples_split=20,
                 max_features="sqrt", bootstrap=True, random_state=42):
        self.n_trees = n_trees
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.max_features = max_features
        self.bootstrap = bootstrap
        self.random_state = random_state
        self.trees = []
        
    def fit(self, X, y):
        np.random.seed(self.random_state)
        n_samples = X.shape[0]
        
        for i in range(self.n_trees):
            if self.bootstrap:
                idx = np.random.choice(n_samples, size=n_samples, replace=True)
                X_boot = X[idx]
                y_boot = y[idx]
            else:
                X_boot, y_boot = X, y
            
            tree = DecisionTree(
                max_depth=self.max_depth,
                min_samples_split=self.min_samples_split,
                max_features=self.max_features,
                random_state=self.random_state + i if self.random_state else None
            )
            tree.fit(X_boot, y_boot)
            self.trees.append(tree)
            
    def predict(self, X):
        predictions = np.array([tree.predict(X) for tree in self.trees])
        return np.apply_along_axis(lambda x: Counter(x).most_common(1)[0][0], axis=0, arr=predictions)
    
    def predict_proba(self, X):
        predictions = np.array([tree.predict(X) for tree in self.trees])
        proba = np.mean(predictions == 1, axis=0)
        return np.vstack([1 - proba, proba]).T


# ================================================
#  Chargement et ingénierie des features (datasetv6)
# ================================================

print("Chargement des données...")
df = pd.read_csv("datasetv6.csv", parse_dates=["Date"])
df = df.sort_values(["ticker", "Date"]).reset_index(drop=True)

# Target : direction du rendement brut 1 jour
df["target"] = (df["decision(rendement_brut_1j)"] > 0).astype(int)

# ==================== Feature Engineering ====================

# Lags importants
lags = [1, 2, 3, 5]
for lag in lags:
    df[f'rendement_lag_{lag}'] = df.groupby('ticker')['Rendement_4'].shift(lag)
    df[f'RSI_14_lag_{lag}'] = df.groupby('ticker')['RSI_14'].shift(lag)
    df[f'MACD_lag_{lag}'] = df.groupby('ticker')['MACD_12_26_9'].shift(lag)

# Features dérivées
df['RSI_diff'] = df['RSI_14'] - df['RSI_14'].shift(1)
df['MACD_histogram'] = df['MACD_12_26_9'] - df['MACDs_12_26_9']
df['SMA_ratio'] = df['SMA_20'] / df['SMA_50']
df['ATR_ratio'] = df['ATR_14'] / df['ATR_4']

# Volatilité rolling
df['vol_10'] = df.groupby('ticker')['Rendement_4'].transform(lambda x: x.rolling(10).std())

# Liste finale des features
features = [
    'RSI_14', 'RSI_4', 'Rendement_14', 'Rendement_4',
    'MACD_12_26_9', 'ATR_14', 'ATR_4',
    'SMA_20', 'SMA_50', 'Volume_norm_20',
    'SMA_ratio', 'ATR_ratio', 'RSI_diff', 'MACD_histogram',
    'vol_10'
] + [f'rendement_lag_{lag}' for lag in lags] \
  + [f'RSI_14_lag_{lag}' for lag in lags] \
  + [f'MACD_lag_{lag}' for lag in lags]

# Préparation des matrices
X = df[features].values
y = df["target"].values

# Suppression des NaN
mask = ~np.isnan(X).any(axis=1)
X = X[mask]
y = y[mask]

print(f"Échantillons après nettoyage : {len(X):,}")
print(f"Distribution target → 0: {sum(y==0)} | 1: {sum(y==1)}")

# Split chronologique (80/20)
train_size = int(len(X) * 0.80)
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

# ================================================
#  Entraînement du modèle amélioré
# ================================================

print("\nEntraînement du Random Forest (300 arbres)...")

rf = RandomForest(
    n_trees=300,
    max_depth=12,
    min_samples_split=25,
    max_features=int(math.sqrt(len(features))),   # ou 'sqrt'
    bootstrap=True,
    random_state=42
)

rf.fit(X_train, y_train)

# ================================================
#  Évaluation
# ================================================

from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

y_pred = rf.predict(X_test)
y_proba = rf.predict_proba(X_test)[:, 1]

print("\n=== RÉSULTATS ===")
print(f"Accuracy  : {accuracy_score(y_test, y_pred):.4f}")
print(f"ROC AUC   : {roc_auc_score(y_test, y_proba):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

# Feature Importance (approximation avec les premiers arbres)
print("\nTop 10 Features les plus importantes :")
# Cette partie est approximative car on n'a pas de feature_importances_ natif
# On peut l'améliorer plus tard avec XGBoost si besoin

Chargement des données...
Échantillons après nettoyage : 78,170
Distribution target → 0: 38157 | 1: 40013

Entraînement du Random Forest (300 arbres)...

=== RÉSULTATS ===
Accuracy  : 0.5152
ROC AUC   : 0.5155

Classification Report:
              precision    recall  f1-score   support

           0       0.52      0.18      0.26      7663
           1       0.52      0.84      0.64      7971

    accuracy                           0.52     15634
   macro avg       0.52      0.51      0.45     15634
weighted avg       0.52      0.52      0.45     15634


Top 10 Features les plus importantes :


## Fonctions auxiliaires
entropy(y) : 
Calcule l’entropie d’un vecteur de labels (0 ou 1).
C’est une mesure d’« impureté » : plus l’entropie est élevée, plus les classes sont mélangées.

information_gain(X_col, y, threshold) :
Calcule le gain d’information obtenu en divisant les données selon un seuil sur une feature.

Formule classique :
- Gain = Entropie_parent - (Entropie_gauche + Entropie_droite pondérées)
- best_split(X, y, features, min_samples_split=5)

Parcourt un sous-ensemble de features et, pour chacune, teste plusieurs seuils possibles pour trouver la meilleure division (celle qui maximise le gain d’information).
Pour accélérer, si une feature a trop de valeurs uniques, il n’en teste qu’une quarantaine (via quantiles).

## Classe DecisionTree (Arbre de décision)

max_depth : profondeur maximale de l’arbre
min_samples_split : nombre minimum d’échantillons pour continuer à diviser
max_features : nombre de features tirées aléatoirement à chaque split (comme dans un vrai Random Forest)

### Méthode _grow_tree :

Si on atteint un critère d’arrêt (profondeur max, trop peu d’échantillons, ou une seule classe) → on crée une feuille avec la classe majoritaire.
Sinon :
On tire aléatoirement un sous-ensemble de features (max_features)
On trouve le meilleur split avec best_split
On divise les données en gauche (<= threshold) et droite (> threshold)
On construit récursivement les deux sous-arbres


### Méthodes predict_one et predict :

Pour prédire, on descend dans l’arbre selon les seuils jusqu’à atteindre une feuille.

## Classe RandomForest (Forêt Aléatoire)

n_trees=50 : nombre d’arbres
bootstrap=True : chaque arbre est entraîné sur un échantillon bootstrap (tirage avec remise)
max_features="sqrt" : chaque arbre utilise seulement √(nombre de features) à chaque split

### Méthode fit :

Tire un échantillon bootstrap des données (si bootstrap=True)
Crée et entraîne un DecisionTree sur ces données

Tous les arbres sont stockés dans self.trees

### Méthodes de prédiction :

predict() : chaque arbre vote → on prend le vote majoritaire
predict_proba() : proportion d’arbres qui prédisent la classe 1 (utile pour le ROC AUC)

## Partie "Exemple d’utilisation" (la plus importante)

### Chargement des données
- Lit datasetv3.csv
- Trie par ticker et par date

### Création de la cible
- target = 1 si decision > 0, sinon 0
(c’est donc un problème de classification binaire : prédire si la décision est positive ou non)

### Sélection des featuresPython'RSI(4)', 'RSI(14)', 'rendement(4)', 'rendement(14)', 'MACD',
'ATR(14)', 'ATR(4)', 'SMA(20)', 'SMA(50)', 'Ratio[SMA(20)/SMA(50)]',
'Volume_norm(20)'

### Nettoyage
- Supprime les lignes contenant des NaN

### Split train/test
- Split chronologique simple : 80% premieres lignes pour l’entraînement, 20% suivantes pour le test
(remarque dans le code : "better to do walk-forward" → c’est vrai, ce split est un peu naïf pour des données temporelles)

### Entraînement du Random Forest
60 arbres
profondeur max = 9
min_samples_split = 12
max_features = sqrt(11) ≈ 3 → très important pour le côté "random" de la forêt
bootstrap activé

### Évaluation
Prédictions sur le test set
Calcul de l’Accuracy et du ROC AUC